In [162]:
import os
import lda
import random

import pandas as pd
import numpy as np

from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## Fetch narratives

In [144]:
# list of files
narr_files = [
    "gpt-4o-mini_0.0_given_local.json",
    "gpt-4o-mini_0.5_given_local.json",
    "gpt-4o-mini_1.2_given_local.json"
]
narr_dir = "data/narratives/"

def load_json_narratives_to_df(narr_dir, narr_files):
    df_list = [
        format_narrative_df(pd.read_json(os.path.join(narr_dir,file)), file)
        for file in tqdm(narr_files, desc="Loading narrative file")
    ]
    narr_df = pd.concat(df_list, ignore_index=True)

    return narr_df

def format_narrative_df(df, file_name):
    print(f"Formatting: {file_name}")
    df_formatted = df.melt(var_name='profile_id', value_name='text')
    df_formatted['model'] = file_name.split(sep='_')[0]
    df_formatted['temperature'] = float(file_name.split(sep='_')[1])
    df_formatted['scenario'] = file_name.split(sep='_')[2]
    df_formatted['sample'] = df_formatted.groupby('profile_id').cumcount() + 1

    return df_formatted

narrs = load_json_narratives_to_df(narr_dir, narr_files)

Loading narrative file:  33%|███▎      | 1/3 [00:00<00:01,  1.67it/s]

Formatting: gpt-4o-mini_0.0_given_local.json


Loading narrative file:  67%|██████▋   | 2/3 [00:01<00:00,  1.72it/s]

Formatting: gpt-4o-mini_0.5_given_local.json


Loading narrative file: 100%|██████████| 3/3 [00:01<00:00,  1.68it/s]

Formatting: gpt-4o-mini_1.2_given_local.json


In [145]:
narrs

,profile_id,text,model,temperature,scenario,sample
0,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,1
1,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,2
2,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,3
3,1,"As the sun began to rise over the barracks, Pr...",gpt-4o-mini,0.0,given,4
4,1,"As the sun began to rise over the barracks, Ma...",gpt-4o-mini,0.0,given,5
...,...,...,...,...,...,...
63355,1056,Samantha Caldwell stared out the window of her...,gpt-4o-mini,1.2,given,16
63356,1056,The sun slipped behind the horizon as Ava Morg...,gpt-4o-mini,1.2,given,17
63357,1056,Standing on the uneven wooden deck of her mode...,gpt-4o-mini,1.2,given,18
63358,1056,"At the edge of dawn, Clara stands by her kitch...",gpt-4o-mini,1.2,given,19


## Preprocessing

In [171]:
count_vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w\w+\b",       # regular expression to split documents into tokens
                                   lowercase=True,                       # convert all characters to lower-case
                                   strip_accents="ascii",                # remove non-ascii characters (WARNING: this is dangerous in Romance languages)
                                   stop_words="english",                 # remove stopwords from a built-in list. We can also provide our own list
                                   ngram_range=(1, 1),                   # generate only unigrams
                                   analyzer="word",                      # build matrix at the word-level 
                                   max_df=0.7,                           # ignore tokens that have a higher document frequency (can be int or percent)
                                   min_df=20,                            # ignore tokens that have a lower document frequency (can be int or percent)
                                   max_features=None,                    # we could impose a maximum number of vocabulary terms
                                   )     

In [172]:
doc_term_matrix = count_vectorizer.fit_transform(narrs["text"])
print(f"Document-term matrix created with shape: {doc_term_matrix.shape}") 

Document-term matrix created with shape: (63360, 11039)


In [159]:
# we can explore the vocaburaly as pairs of (word, position in matrix)
random.sample(list(count_vectorizer.vocabulary_.items()),10)

[('historically', 4620),
 ('bump', 1153),
 ('successes', 9451),
 ('nipped', 6432),
 ('landscaping', 5412),
 ('palette', 6758),
 ('ether', 3359),
 ('bloomed', 918),
 ('tightly', 9911),
 ('defiant', 2463)]

## LDA

In [160]:
model = lda.LDA(n_topics=15, n_iter=1500, random_state=1)

In [161]:
model.fit(doc_term_matrix)

INFO:lda:n_documents: 63360
INFO:lda:vocab_size: 11041
INFO:lda:n_words: 8452743
INFO:lda:n_topics: 15
INFO:lda:n_iter: 1500
Loading narrative file:   0%|          | 0/3 [19:26<?, ?it/s]
INFO:lda:<0> log likelihood: -90484425
INFO:lda:<10> log likelihood: -77418211
INFO:lda:<20> log likelihood: -71174220
INFO:lda:<30> log likelihood: -69722002
INFO:lda:<40> log likelihood: -69015009
INFO:lda:<50> log likelihood: -68648322
INFO:lda:<60> log likelihood: -68443204
INFO:lda:<70> log likelihood: -68303311
INFO:lda:<80> log likelihood: -68168861
INFO:lda:<90> log likelihood: -68107732
INFO:lda:<100> log likelihood: -68046409
INFO:lda:<110> log likelihood: -68028582
INFO:lda:<120> log likelihood: -68007789
INFO:lda:<130> log likelihood: -67981760
INFO:lda:<140> log likelihood: -67959779
INFO:lda:<150> log likelihood: -67949884
INFO:lda:<160> log likelihood: -67949729
INFO:lda:<170> log likelihood: -67927529
INFO:lda:<180> log likelihood: -67929805
INFO:lda:<190> log likelihood: -67917792
INFO

In [170]:
n_top_words = 10

for i, topic_dist in enumerate(model.topic_word_):
  topic_words = np.array(list(count_vectorizer.vocabulary_.keys()))[np.argsort(topic_dist)][:-(n_top_words+1):-1]
  print('Topic {}: {}'.format(i, ' '.join(topic_words)))

Topic 0: tractor thirty coffees mundane whipped sputtering documented beige traditionally dont
Topic 1: coffees crumbling exposure validation tablet documented sputtering tamales cilantro talisman
Topic 2: interrupted vocation makes hopenot astronaut documented diffused sputtering squash staff
Topic 3: documented whipped leo coffees staff stale intertwine tainted sculpting sunflowers
Topic 4: journeyfilled grass whipped honed mundane reconnect documented staff exposure reminiscing
Topic 5: forging dazzling whipped tailoring unworthy coursework sculpting unfettered coffees intently
Topic 6: mundane threatens towering alive whipped revolutionizing blanketed coffees declined smeared
Topic 7: creatives exposure mundane nested rituals attached makers summoned stepping engages
Topic 8: sunflowers afloat intertwine documented blanketed coffees staff sputtering monitor exposure
Topic 9: sail documented whipped coffees hadn glare summoned walks defied intertwine
Topic 10: journeyfilled alive in